# Exercise 4 – Task: a new scene with Ueli and Bruno

**Goal:** a new scene of your own – *Ueli rows a small red rowboat across the lake, Bruno sits in the bow.* Write its Google Flow prompt, create the clip in Google Flow and measure whether Ueli and Bruno keep their look: **beanie, jacket and dog**.

Build it yourself in three steps. Every step can be done by **copying code from `video_generation_example.ipynb` and adapting it**.

Each step: write your code in the empty cell → run the **check** (✓ go on, ✗ shows what is missing; a `NameError` means the variable does not exist yet). Stuck or out of time? Open the **hint**, then the **solution**.

## Setup

Select the **Python 3.11** kernel. You need two things for this exercise:

- **OpenAI API key** – for the scene prompt (step 1), your own character sheet if you want one, and the consistency check (step 3). It comes from your `OPENAI_API_KEY` Codespaces secret. This notebook uses the **OpenAI API** (model `gpt-6-luna`); other providers have their own APIs with different code.
- **Google account** – to sign in to Google Flow for the video. You create the clip by hand in the Google Flow web app, not in this notebook. One clip costs 10 of your 50 daily credits. No Google Flow clip? The notebook then uses Ueli's clip from Exercise 2.

In [ ]:
# Python libraries.
import sys
from pathlib import Path
from typing import Literal

import pandas as pd
from pydantic import BaseModel

# helper.py and checks.py are in the main folder, one level up.
sys.path.append("..")
from helper import DATA, ask, create_pictures, extract_frames, show_images
from checks import check_consistency, check_frame_paths, check_scene

## Step 1: the prompt for the new scene

Write the Google Flow prompt for the scene: **Ueli rows a small red rowboat across a mountain lake, Bruno sits in the bow.** Use the same `CHARACTER` and `DOG` descriptions and the same five parts as in the example. Store the model's answer in `plan` and the complete prompt in `flow_prompt`.

In [ ]:
# Your code here.


In [ ]:
check_scene(plan, flow_prompt, CHARACTER, DOG)

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hint</summary>

Example, section 1 – copy it and change only the line that describes the scene.

</details>

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Solution</summary>

```python
CHARACTER = "Ueli, red beanie, olive jacket"
DOG = "Bruno, Bernese, red collar"


class ScenePrompt(BaseModel):
    shot: str
    camera: str
    action: str
    light: str
    sound: str


PROMPT = f"""
Write one short video prompt for Google Flow.
The scene: {CHARACTER} rows a small red rowboat across a mountain lake,
the dog {DOG} sits in the bow.
Describe shot, camera, action, light and sound, each as one short sentence.
"""

plan = ask(PROMPT, ScenePrompt, [])
flow_prompt = f"{CHARACTER}. {DOG}. " + " ".join(plan.values())
print(flow_prompt)
```

</details>

## Google Flow: create the clip (by hand)

The character sheet is the reference picture: for Google Flow and for the check in step 3. Use the supplied sheet `ueli_sheet.jpg`, or set `OWN_SHEET = True` to let the OpenAI API create your own sheet from `CHARACTER` and `DOG` (model `gpt-image-2`, about US$0.006). It is saved as `reference/my_sheet.png`.

In [ ]:
# The character sheet: the supplied one, or your own from the OpenAI API.
OWN_SHEET = False

if OWN_SHEET:
    sheet_prompt = (
        f"Character reference sheet of {CHARACTER}, together with the dog {DOG}. "
        "Front view and side view of both, full body, plain white background, "
        "photorealistic."
    )
    sheet = create_pictures(["my_sheet"], [sheet_prompt], "reference")[0]
else:
    sheet = DATA / "reference" / "ueli_sheet.jpg"
show_images([sheet])

**Now in the Google Flow web app** ([labs.google/flow](https://labs.google/flow)): use **Veo 3.1 Lite**, 8 seconds and **Ingredients to Video**. Add your character sheet as ingredient and paste `flow_prompt` from step 1. Download the clip and upload it as `my_scene.mp4` next to this notebook.

## Step 2: frames of your clip

Use your clip `my_scene.mp4`. If it does not exist, use Ueli's clip `ueli_scene3_talk.mp4` from `data/videos/` – and then also the supplied sheet, which shows the same Ueli. Save one frame per second in the folder `frames_clip` and store the file paths in `paths`.

In [ ]:
# Your code here.


In [ ]:
check_frame_paths(paths)

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hint</summary>

Example, section 3: `extract_frames(video_path, folder="frames_clip")`. Check for your clip with `Path("my_scene.mp4").exists()`.

</details>

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Solution</summary>

```python
video_path = Path("my_scene.mp4")
if not video_path.exists():
    video_path = DATA / "videos" / "ueli_scene3_talk.mp4"
    sheet = DATA / "reference" / "ueli_sheet.jpg"
print("Using", video_path.name, "and", Path(sheet).name)

paths = extract_frames(video_path, folder="frames_clip")
show_images(paths)
```

</details>

## Step 3: beanie, jacket and dog

Compare every frame with the sheet – now for **three** features: `beanie`, `jacket` and `dog`. Build the DataFrame `ratings` with one row per frame and the column `frame`, then print for each feature how many frames match.

In [ ]:
# Your code here.


In [ ]:
check_consistency(ratings, paths)

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hint</summary>

Example, section 3: the same loop – your class gets three fields instead of one, and the prompt names all three features.

</details>

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Solution</summary>

```python
Rating = Literal["match", "differs", "not visible"]


class Consistency(BaseModel):
    beanie: Rating
    jacket: Rating
    dog: Rating


PROMPT = """
Image 1 is the reference. Rate the beanie, the jacket and the dog in image 2.
Use match for a visible match, differs for a visible change
and not visible when it is unclear or out of frame.
"""

rows = []
for path in paths:
    row = ask(PROMPT, Consistency, [sheet, path])
    row["frame"] = path.name
    rows.append(row)

ratings = pd.DataFrame(rows)
for feature in ["beanie", "jacket", "dog"]:
    print(feature, ":", (ratings[feature] == "match").sum(), "of", len(ratings), "frames match")
ratings
```

</details>

## Check your understanding

Answer briefly using the outputs above. Try first, then open the hints.

1. If every frame were generated as a separate image, the video would flicker (details jump from frame to frame). Why – and how do video models avoid it?
2. Why do we put the exact character description in front of the prompt, together with the character sheet?
3. Which feature matched least often in your clip? Look at these frames: is the character really different, or is the feature just not visible?
4. The ratings come from a model. How would you check whether you can trust them?
5. Why is the number of matching frames a better result than "the clip looks fine"?

**Your answers:**

1. …
2. …
3. …
4. …
5. …

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hints for self-check</summary>

1. Every image starts from new random noise, so details like the face or the clothes come out differently each time. Video models generate all frames together, across space and time. That keeps the details stable.

2. Small changes in the wording can produce a different person or dog. The same text plus the same reference image anchor the identity – also when you create more clips of Ueli later.

3. Use your table. `not visible` is not an error of the video: Ueli may be turned away or out of frame. Only `differs` shows that the character has changed.

4. Rate a few frames yourself and compare them with the ratings of the model. Where you and the model disagree, the model is not reliable.

5. A number can be compared between clips, prompts and settings, and other people can repeat the measurement. An impression cannot be compared.

</details>

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')